# Task 2：PySpark Data Preparation

## 1. Imports and Configuration

In [6]:
from pathlib import Path
import sys, json, os
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

In [7]:
ROOT = next((candidate for base in [Path.cwd(), *Path.cwd().parents]
             for candidate in [base, base / 'data_prep', base / 'task2/data_prep']
             if (candidate / 'src/build_task2_dataset.py').is_file()), None)
assert ROOT is not None, 'Cannot find task2/data_prep; check the working directory'
assert (ROOT / 'src/build_task2_dataset.py').is_file(), 'Please open the Notebook of this path'
sys.path.insert(0, str(ROOT / 'src'))
from build_task2_dataset import build_examples

INPUT = Path('/scratch/e0969984/singapore-rainfall-analysis-and-prediction/data/processed')

THRESHOLDS = [0., 1., 5., 10.]
FEATURES = ['station_id', 'hour', 'month', 'past_5m_mm', 'past_30m_mm', 'past_60m_mm', 'threshold_mm']
OUTPUT = ROOT.parent / 'data/task2' / datetime.now(timezone.utc).strftime('notebook_%Y%m%dT%H%M%SZ')
print('输出:', OUTPUT)

输出: /nfs/home/svu/le.pan/singapore-rainfall-analysis-and-prediction/task2/data/task2/notebook_20261014T073533Z


## 2. Initialize Spark Session

In [8]:
import socket
import os

print("Hostname:", socket.gethostname())
print("CPU count:", os.cpu_count())
print("Working directory:", os.getcwd())

Hostname: CN-149
CPU count: 72
Working directory: /nfs/home/svu/le.pan/singapore-rainfall-analysis-and-prediction/task2/notebooks


In [9]:
spark = (SparkSession.builder.master('local[4]').appName('Task2Notebook')
         .config('spark.sql.session.timeZone', 'Asia/Singapore')
         .config('spark.sql.shuffle.partitions', '128').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Python:', sys.version, '\nSpark:', spark.version, '\nSpark UI (计算节点):', spark.sparkContext.uiWebUrl)

Python: 3.11.17 (main, Oct 14 2026, 01:05:33) [Clang 23.1.3 ] 
Spark: 3.5.6 
Spark UI (计算节点): http://CN-149.stdct.loc:4040


## 3. Load Rainfall Data (2017-2024)

In [10]:

YEARS = list(range(2017, 2025))
frames = []

for year in YEARS:
    path = INPUT / f'rainfall_{year}'
    assert path.is_dir(), f'Path does not exist or is not accessible: {path}'

    frame = spark.read.parquet(str(path)).select(
        'station_id', 'timestamp', 'rainfall'
    )

    assert frame.where(
        F.year('timestamp') != year
    ).limit(1).count() == 0

    frames.append(frame)

observations = frames[0]

for frame in frames[1:]:
    observations = observations.unionByName(frame)

observations.printSchema()
observations.show(10, truncate=False)


root
 |-- station_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- rainfall: double (nullable = true)

+----------+-------------------+--------+
|station_id|timestamp          |rainfall|
+----------+-------------------+--------+
|S06       |2017-01-01 08:25:00|0.0     |
|S06       |2017-01-01 08:40:00|0.0     |
|S06       |2017-01-01 08:45:00|0.0     |
|S06       |2017-01-01 09:00:00|0.0     |
|S06       |2017-01-01 09:30:00|0.0     |
|S06       |2017-01-01 10:45:00|0.0     |
|S06       |2017-01-01 10:55:00|0.0     |
|S06       |2017-01-01 11:25:00|0.0     |
|S06       |2017-01-01 11:30:00|0.0     |
|S06       |2017-01-01 11:45:00|0.0     |
+----------+-------------------+--------+
only showing top 10 rows



## 4. Validate Input Data Quality

In [11]:
invalid = (F.col('station_id').isNull() | (F.trim('station_id') == '') |
           F.col('timestamp').isNull() | (F.col('timestamp').cast('long') % 300 != 0) |
           (F.col('timestamp').cast('double') != F.col('timestamp').cast('long')) |
           F.col('rainfall').isNull() | F.isnan('rainfall') | (F.col('rainfall') < 0) |
           (F.abs('rainfall') == float('inf')))
assert observations.where(invalid).limit(1).count() == 0, 'Invalid Observation'
assert observations.groupBy('station_id','timestamp').count().where('count > 1').limit(1).count() == 0, 'Repeat Observation'
print('Pass.')

26/10/14 15:42:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/14 15:42:31 WARN RowBasedKeyValueBatch: Calling spill() on

Pass.


## 5. Construct Historical and Future Rainfall Windows

In [12]:
audit, examples = build_examples(observations, THRESHOLDS)
audit = audit.persist(StorageLevel.DISK_ONLY)
examples = examples.persist(StorageLevel.DISK_ONLY)
examples.show(10, truncate=False)

[Stage 48:=====================================================>(127 + 1) / 128]

+----------+-------------------+----+-----+----+-----+----------+-----------+-----------+------------+-----+-------------+
|station_id|timestamp          |year|split|hour|month|past_5m_mm|past_30m_mm|past_60m_mm|threshold_mm|label|future_30m_mm|
+----------+-------------------+----+-----+----+-----+----------+-----------+-----------+------------+-----+-------------+
|S208      |2020-06-15 13:45:00|2020|train|13  |6    |0.204     |0.204      |0.204      |0.0         |1.0  |0.204        |
|S208      |2020-06-15 13:45:00|2020|train|13  |6    |0.204     |0.204      |0.204      |1.0         |0.0  |0.204        |
|S208      |2020-06-15 13:45:00|2020|train|13  |6    |0.204     |0.204      |0.204      |5.0         |0.0  |0.204        |
|S208      |2020-06-15 13:45:00|2020|train|13  |6    |0.204     |0.204      |0.204      |10.0        |0.0  |0.204        |
|S208      |2020-06-15 13:50:00|2020|train|13  |6    |0.204     |0.408      |0.408      |0.0         |0.0  |0.0          |
|S208      |2020

## 6. Assess Data Completeness and Class Balance

In [13]:
quality = audit.groupBy('year').agg(
    F.count('*').alias('observations'),
    F.sum((~F.col('history_complete')).cast('long')).alias('incomplete_history'),
    F.sum((~F.col('future_complete')).cast('long')).alias('incomplete_future'),
    F.sum((F.col('history_complete') & F.col('future_complete') & ~F.col('split_safe')).cast('long')).alias('boundary_excluded'),
    F.sum((F.col('history_complete') & F.col('future_complete') & F.col('split_safe')).cast('long')).alias('eligible_times'))
quality.orderBy('year').show(8, truncate=False)
balance = examples.groupBy('split', 'threshold_mm').agg(
    F.count('*').alias('samples'), F.sum('label').alias('positive_samples'), F.avg('label').alias('positive_rate'))
balance.orderBy('split', 'threshold_mm').show(12, truncate=False)

+----+------------+------------------+-----------------+-----------------+--------------+
|year|observations|incomplete_history|incomplete_future|boundary_excluded|eligible_times|
+----+------------+------------------+-----------------+-----------------+--------------+
|2017|5256020     |314040            |198461           |0                |4822618       |
|2018|4594246     |565923            |514505           |0                |3981287       |
|2019|5026862     |61753             |35516            |0                |4935783       |
|2020|6334473     |69880             |42187            |0                |6234155       |
|2021|7044646     |49498             |31191            |0                |6976422       |
|2022|7133230     |41582             |24912            |420              |7072758       |
|2023|6742890     |38729             |23238            |378              |6686625       |
|2024|6406260     |49389             |31784            |0                |6338426       |
+----+----

[Stage 57:=====================================================>(127 + 1) / 128]

+----------+------------+--------+----------------+---------------------+
|split     |threshold_mm|samples |positive_samples|positive_rate        |
+----------+------------+--------+----------------+---------------------+
|test      |0.0         |6338426 |403195.0        |0.0636112183056172   |
|test      |1.0         |6338426 |162937.0        |0.025706224226645542 |
|test      |5.0         |6338426 |53122.0         |0.008380945048502578 |
|test      |10.0        |6338426 |24265.0         |0.0038282374835645317|
|train     |0.0         |34023023|2043367.0       |0.06005836106920893  |
|train     |1.0         |34023023|793621.0        |0.023325998986039542 |
|train     |5.0         |34023023|277109.0        |0.00814474951270497  |
|train     |10.0        |34023023|132687.0        |0.003899918005522319 |
|validation|0.0         |6686625 |446087.0        |0.06671332697736153  |
|validation|1.0         |6686625 |175835.0        |0.02629652477894304  |
|validation|5.0         |6686625 |5843

## 7. Validate and Save the ML Dataset

In [14]:

summary = [r.asDict() for r in balance.collect()]

assert {r['split'] for r in summary} == {
    'train', 'validation', 'test'
}

eligible = sum(r.eligible_times for r in quality.collect())

assert sum(r['samples'] for r in summary) == eligible * len(THRESHOLDS)
assert eligible > 0

assert not OUTPUT.exists(), 'Please use the new output path'
OUTPUT.mkdir(parents=True)

manifest = {
    'status': 'running',
    'input_root': str(INPUT),
    'features': FEATURES,
    'thresholds_mm': THRESHOLDS,
    'timezone': 'Asia/Singapore',
    'spark_version': spark.version,
    'class_balance': summary,
    'history': 't-55 through t: 12 complete five-minute observations',
    'label': 'sum(t+5 through t+30) > threshold; 6 complete observations',
    'split': {
        'train': '2017-2022',
        'validation': '2023',
        'test': '2024'
    },
    'missingness': 'exclude incomplete windows; no zero imputation',
    'forbidden_features': ['future_30m_mm', 'label', 'split']
}

manifest_file = OUTPUT / 'manifest.json'
manifest_file.write_text(
    json.dumps(manifest, indent=2),
    encoding='utf-8'
)

examples.write.mode('errorifexists') \
    .partitionBy('split', 'year') \
    .parquet(str(OUTPUT / 'examples'))

quality.orderBy('year').toPandas().to_csv(
    OUTPUT / 'quality_by_year.csv', index=False
)

balance.orderBy('split', 'threshold_mm').toPandas().to_csv(
    OUTPUT / 'class_balance.csv', index=False
)

manifest['status'] = 'complete'
manifest_file.write_text(
    json.dumps(manifest, indent=2),
    encoding='utf-8'
)

print('Save at:', OUTPUT)


[Stage 86:=====================================================>(127 + 1) / 128]

已保存: /nfs/home/svu/le.pan/singapore-rainfall-analysis-and-prediction/task2/data/task2/notebook_20261014T073533Z


In [15]:
# Clean up Spark Resources
examples.unpersist()
audit.unpersist()
spark.stop()

26/10/14 16:01:07 WARN BlockManager: Block rdd_198_120 could not be removed as it was not found on disk or in memory
26/10/14 16:01:07 WARN BlockManager: Block rdd_198_103 could not be removed as it was not found on disk or in memory
26/10/14 16:01:07 WARN BlockManager: Block rdd_198_4 could not be removed as it was not found on disk or in memory
26/10/14 16:01:07 WARN BlockManager: Block rdd_198_50 could not be removed as it was not found on disk or in memory
